# 🤖 شات بوت بذاكرة واستدعاء أدوات (Chatbot with Memory & Tool Use)
### مشروع D3 — مسار الذكاء الاصطناعي التوليدي (GenAI Track)

---
## 🎯 المشكلة التجارية (Business Problem)
متجر عايز **مساعد محادثة** يرد على العملاء، بس الردود لازم تكون مبنية على **بيانات حقيقية ولحظية**
(حالة طلب، سعر منتج، توفّره) مش على "كلام عام". والمساعد لازم **يفتكر** سياق المحادثة
(مثلاً: "الطلب ده هيوصل إمتى؟" بعد ما سأل عن طلب معيّن).

**الحل:** **استدعاء الأدوات (Tool Use / Function Calling)** — النموذج بيقرّر يستدعي دالة (أداة)،
ندّيله نتيجتها، ويصيغ الرد. + **ذاكرة محادثة (Conversation Memory)** تحافظ على السياق.

## 📦 ما الذي يثبته المشروع
تعريف الأدوات (Tool schemas) · **حلقة استدعاء الأدوات (Agentic loop)** · **ذاكرة المحادثة** ·
نسخة أوفلاين شغّالة بالكامل + نسخة إنتاج بـ **Anthropic API الحقيقي**.


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/Ahmedelgendyyy/data-science-portfolio"
PROJECT = "genai/d3_chatbot_tools"          # مسار المشروع داخل portfolio/
PKGS    = ["anthropic"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| **Function Calling / Tool Use** | وثائق Anthropic (Tool Use) | النموذج يستدعي كودك لجلب بيانات/تنفيذ فعل |
| Tool schema (JSON Schema) | وثائق Anthropic | وصف الأداة ومدخلاتها للنموذج |
| **الحلقة الوكيلة (Agentic loop)** | Anthropic · Huyen (ch. agents) | request → tool_use → نفّذ → tool_result → رد |
| ذاكرة المحادثة (stateless API) | وثائق Anthropic (Multi-turn) | الـ API بلا حالة → تبعت التاريخ كامل كل مرة |
| تصميم الأدوات | shared/tool-use-concepts | أوصاف واضحة + متى تُستدعى |

> 🎯 **بيُستخدم في الواقع:** شات بوتس خدمة العملاء، المساعدين الشخصيين، الـ agents اللي بتنفّذ مهام.
> 🛠️ **هنا:** نبني الأدوات والذاكرة ونشغّلها **أوفلاين** بموجّه بسيط، والخلية الأخيرة تستخدم
> **Claude tool-use API الحقيقي** (محاطة بـ try/except عشان ما تكسرش النوتبوك بدون مفتاح).


## 0️⃣ المكتبات وبيانات المتجر

In [ ]:
import json, re, os
with open('data/shop_data.json', encoding='utf-8') as f:
    SHOP = json.load(f)
# TODO: جهّز PRODUCTS (dict بالاسم) و ORDERS و STORE
PRODUCTS = ...
ORDERS = SHOP['orders']
STORE = SHOP['store']
print(list(PRODUCTS))

## 1️⃣ الأدوات (Tools) — دوال بايثون حقيقية
كل أداة دالة بترجّع نص. دي الكود اللي النموذج "بيستدعيه" لمّا يحتاج معلومة حقيقية.

In [ ]:
# TODO: عرّف الأدوات: get_order_status, get_product_price, check_stock, calculate, store_info
def get_order_status(order_id): ...
def get_product_price(product_name): ...
def check_stock(product_name): ...
def calculate(expression): ...
def store_info(): ...
DISPATCH = {...}
print(get_order_status('ORD1001'))

## 2️⃣ مخطّطات الأدوات (Tool Schemas — صيغة Anthropic)
دي اللي بتتبعت للنموذج عشان يعرف الأدوات المتاحة ومدخلاتها. **الوصف لازم يقول "متى تُستخدم"**.

In [ ]:
# TODO: عرّف TOOLS كقائمة بصيغة Anthropic (name, description, input_schema) لكل أداة
TOOLS = [ ... ]
print(len(TOOLS))

## 3️⃣ شات بوت بذاكرة (offline) — يشتغل بدون مفتاح API
بدل النموذج، موجّه (router) بسيط بيقرّر الأداة من الكلمات المفتاحية. الأهم هنا: **الذاكرة** —
بنحفظ تاريخ المحادثة + سياق (آخر طلب اتكلّمنا عنه) عشان نرد على الأسئلة المتابِعة.

In [ ]:
# TODO: اكتب router(msg, context) + كلاس ToolChatBot فيه history و context
#       استخدم context['last_order'] للأسئلة المتابِعة (الذاكرة)
def route(msg, context): ...
class ToolChatBot: ...
bot = ToolChatBot()
print(bot.send("سعر الـ Laptop كام؟"))

## 4️⃣ (إنتاج) الشات بوت بـ Claude API الحقيقي — حلقة استدعاء الأدوات
دي الحلقة الفعلية: نبعت الرسالة + الأدوات → لو النموذج طلب أداة (`stop_reason == "tool_use"`) ننفّذها
ونرجّع `tool_result` → نكرّر لحد ما يخلّص. محاطة بـ try/except عشان تتخطّى أوفلاين بدون مفتاح.

In [ ]:
def chat_with_claude(user_msg, history=None, model="claude-opus-4-8"):
    """شات بوت إنتاجي باستدعاء أدوات حقيقي. يتطلب ANTHROPIC_API_KEY.
       (للتجارب الأرخص بدّل model إلى "claude-haiku-4-5".)"""
    try:
        import anthropic                                   # pip install anthropic
        client = anthropic.Anthropic()                     # يقرأ ANTHROPIC_API_KEY من البيئة
        messages = list(history or [])
        messages.append({"role": "user", "content": user_msg})

        while True:
            resp = client.messages.create(
                model=model, max_tokens=1024,
                system="أنت مساعد متجر TechNest. استخدم الأدوات لجلب بيانات حقيقية، وردّ بالعربية باختصار.",
                tools=TOOLS, messages=messages)

            if resp.stop_reason != "tool_use":
                return next((b.text for b in resp.content if b.type == "text"), ""), messages

            messages.append({"role": "assistant", "content": resp.content})
            tool_results = []
            for block in resp.content:
                if block.type == "tool_use":
                    out = DISPATCH[block.name](**block.input)        # نفّذ نفس أدواتنا
                    tool_results.append({"type": "tool_result",
                                         "tool_use_id": block.id, "content": str(out)})
            messages.append({"role": "user", "content": tool_results})
    except Exception as e:
        return (f"[تم تخطّي Claude API: {type(e).__name__}] — شغّل النسخة الأوفلاين بالأعلى. "
                f"للإنتاج: pip install anthropic + ضبط ANTHROPIC_API_KEY."), (history or [])

answer, _ = chat_with_claude("الطلب ORD1002 حالته إيه، وإجمالي سعر Laptop + Monitor كام؟")
print(answer)

## 5️⃣ الخلاصة والتوصيات (Conclusion)
- **استدعاء الأدوات (Tool Use)** بيربط النموذج ببيانات/أفعال حقيقية — مفيش "اختراع" معلومات.
- **الذاكرة:** الـ API بلا حالة (stateless)، فبنبعت تاريخ المحادثة كامل كل مرة؛ والسياق (آخر طلب)
  بيخلّي الردود المتابِعة منطقية.
- **الحلقة الوكيلة:** `request → tool_use → نفّذ → tool_result → رد` هي قلب أي **agent**.
- **النسخة الأوفلاين** أثبتت المعمار كامل بدون مفتاح؛ **نسخة الإنتاج** بتستخدم Claude الحقيقي بنفس الأدوات.
- **للإنتاج:** أضف معالجة أخطاء الأدوات (`is_error`)، حدود لعدد اللفّات، وتسجيل (logging)،
  ولو الأدوات كتير استخدم **Tool Search**.

> ✅ **اللي اتعلمته:** تعريف الأدوات، حلقة الـ tool-use، ذاكرة المحادثة، وربط نموذج حقيقي بأدوات بايثون.
